In [1]:
import json
import os 

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "kano2015social")
original_data_pathway = os.path.join(pathway, "original_data")

complete_path_1 = os.path.join(original_data_pathway, "2015_pone.csv")

out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

In [2]:
import pandas as pd
import numpy as np
import pyreadstat

df = pd.read_csv(complete_path_1)

df['study_id']="kano2015social"
df.columns = map(str.lower, df.columns)
df=df.applymap(lambda s: s.lower() if type(s) == str else s)
df.rename(columns={"species": "species_original",
    "name":"ape",
    "sex":"sex_original",
    "rearing h.":"rearing h"}, inplace=True)


In [3]:
comp_path_name_errors = os.path.join(pathway_gen, "common_name_errors.csv")

df_name  = pd.read_csv(comp_path_name_errors)
df['ape'] = df['ape'].str.rstrip()
for x,y in zip(df_name['wrong'],df_name['right']):
    df['ape'].replace(x, y, inplace=True)

comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)    
df= df.merge(apedf,left_on='ape', right_on='name', how='left')

In [4]:
import re
replace_1=re.compile('(\(|\)| )')
df.columns = df.columns.str.replace(replace_1, '_')
df.columns = df.columns.str.replace('__', '_')


In [5]:
# name_correction_list = [['lolita', 'loleta']]
# for x,y in name_correction_list:
#     df['ape'].replace(x, y, inplace=True)



In [6]:

bonobos = ['vijay','lenore', 'ikela', 'louise', 'junior', 'lolita']
chimpanzees = [ 'zamba','iroha', 'misaki', 'mizuki', 'hatsuka', 'natsuki']

spe_2=[] 
for index, row in df.iterrows():
    if row['ape'] in bonobos:
        spe_2.append('bonobo')
    elif row['ape'] in chimpanzees:
        spe_2.append('chimpanzee')
    else:
        spe_2.append(np.nan)
df = df.assign(spe_2=spe_2)

spe_3=[] 
for index, row in df.iterrows():
    if not pd.isna(row['species']):
        spe_3.append(row['species'])
    else:
        spe_3.append(row['spe_2'])
df = df.assign(species=spe_3)

spe_4=[] 
for index, row in df.iterrows():
    if not pd.isna(row['sex']):
        spe_4.append(row['sex'])
    else:
        spe_4.append(row['sex_original'])
df = df.assign(sex=spe_4)


In [7]:

df.rename(columns={"ape": "participant",
                   'eye_ms_':'full-face_trial_eye_ms ', 
                   'mouth_ms_':'full-face_trial_mouth_ms ', 
                   'face_ms_':'full-body_trial_face', 
                   'genital_ms_':'full-body_trial_genital', 
                   'target_ms_':'full-body_trial_target',
                   'age':'age_in_years'}, inplace=True)
# df.columns

In [8]:
kano2015social_standardized=df[['study_id',  'participant','age_in_years', 'sex', 'species', 'facility',
        'full-face_trial_eye_ms ', 'full-face_trial_mouth_ms ',
       'full-body_trial_face', 'full-body_trial_genital',
       'full-body_trial_target',
       'missclassified', 'discriminant_score']]
comp_out_path_stand = os.path.join(out_pathway, 'kano2015social_standardized.csv')
kano2015social_standardized.to_csv(comp_out_path_stand, encoding='utf-8-sig', index=False)


names =kano2015social_standardized.columns.tolist()
df = pd.DataFrame(names)
df = df.rename(columns={0: "column_name"})
df["description"] = ""
kano2015social_glossary=df[["column_name", "description"]]

comp_out_path_glossary = os.path.join(out_pathway, 'kano2015social_glossary.csv')
kano2015social_glossary.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)
